# LoRA Fine-Tuning — Colab Experiment

QLoRA fine-tuning of Mistral-7B on ML interview Q&A.

**Covers:**
1. Data preparation (`win-wang/Machine_Learning_QA_Collection`)
2. Baseline training (r=16, full dataset — ~40 min on A100)
3. Rank sweep: r=8 / r=16 / r=64 at 1000 examples
4. Data scaling: n=100 / n=500 / n=1000 at r=16
5. Perplexity evaluation (base vs fine-tuned)
6. Qualitative side-by-side comparison
7. Save to Google Drive or download

**Required runtime:** A100 — *Runtime → Change runtime type → GPU → A100*

**HuggingFace token:** Add `HF_TOKEN` to Colab Secrets (key icon, left sidebar). Required for Mistral-7B.

In [ ]:
import subprocess, sys
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('No GPU — Runtime > Change runtime type > GPU > A100')
    sys.exit(1)

In [ ]:
%pip install -q transformers datasets peft trl bitsandbytes accelerate huggingface_hub pyyaml

In [ ]:
import os

# Set USE_DRIVE = True to persist checkpoints across sessions.
# False = checkpoints live in /content (lost on session end, but download at end).
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_BASE = '/content/drive/MyDrive/lora-finetune/checkpoints'
else:
    CKPT_BASE = '/content/checkpoints'

os.makedirs(CKPT_BASE, exist_ok=True)
print('Checkpoints base:', CKPT_BASE)

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('Logged in from Colab Secrets')
except Exception:
    login()  # prompts for token

## Data Preparation

Downloads `win-wang/Machine_Learning_QA_Collection` (~8600 Gemma-format conversations),
converts to Alpaca JSONL, writes `train.jsonl` and `val.jsonl` (90/10 split).

Quality filters: skip questions < 20 chars, answers < 30 chars.

In [ ]:
import json, re
from datasets import load_dataset

def parse_gemma(text):
    u = re.search(r'<start_of_turn>user\s*(.*?)<end_of_turn>', text, re.DOTALL)
    m = re.search(r'<start_of_turn>model\s*(.*?)(?:<end_of_turn>|$)', text, re.DOTALL)
    if not u or not m:
        return None
    q, a = u.group(1).strip(), m.group(1).strip()
    return (q, a) if len(q) >= 20 and len(a) >= 30 else None

def write_jsonl(examples, path):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    with open(path, 'w') as f:
        for ex in examples:
            f.write(json.dumps(ex) + '\n')

print('Downloading win-wang/Machine_Learning_QA_Collection...')
ds = load_dataset('win-wang/Machine_Learning_QA_Collection')

examples, skipped = [], 0
for row in ds['train']:
    p = parse_gemma(row['text'])
    if p:
        examples.append({'instruction': p[0], 'input': '', 'output': p[1]})
    else:
        skipped += 1

n_val = max(50, len(examples) // 10)
train_data, val_data = examples[:-n_val], examples[-n_val:]

write_jsonl(train_data, '/content/data/train.jsonl')
write_jsonl(val_data, '/content/data/val.jsonl')

print(f'Train: {len(train_data)}, Val: {len(val_data)}, Skipped: {skipped}')
print('\nSample instruction:', train_data[0]['instruction'][:120])

## Configuration & Helpers

- `load_model(rank)` — loads 4-bit Mistral-7B with LoRA adapters (0.5% of params trained)
- `run_train(model, tok, out_dir, n, epochs)` — SFTTrainer, saves adapter to `out_dir/final`
- `compute_ppl(model, tok)` — average perplexity on val set (lower = better fit)

In [ ]:
import math, torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import (
    LoraConfig, TaskType, get_peft_model,
    prepare_model_for_kbit_training, PeftModel
)
from trl import SFTTrainer
from datasets import load_dataset as lds

BASE_MODEL = 'mistralai/Mistral-7B-v0.1'
SYSTEM = (
    'You are an expert ML engineer. Answer the following question clearly '
    'and concisely, as you would in a technical interview.'
)

def fmt(ex):
    return (
        '<s>[INST] <<SYS>>\n' + SYSTEM + '\n<</SYS>>\n\n'
        + '### Instruction:\n' + ex['instruction'] + '\n\n'
        + '### Response:\n' + ex['output'] + ' [/INST]</s>'
    )

def load_model(rank=16):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4',
    )
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    tok.padding_side = 'right'
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=rank, lora_alpha=rank * 2,
        lora_dropout=0.05, target_modules=['q_proj', 'v_proj'], bias='none',
    ))
    model.print_trainable_parameters()
    return model, tok

def run_train(model, tok, out_dir, n=None, epochs=3):
    ds = lds('json', data_files='/content/data/train.jsonl', split='train')
    if n:
        ds = ds.select(range(min(n, len(ds))))
    print(f'  Training on {len(ds)} examples for {epochs} epochs...')
    trainer = SFTTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=out_dir, num_train_epochs=epochs,
            per_device_train_batch_size=4, gradient_accumulation_steps=4,
            learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.05,
            fp16=True, logging_steps=10, save_strategy='epoch', save_total_limit=1,
            report_to='none', optim='paged_adamw_32bit', group_by_length=True,
        ),
        train_dataset=ds, formatting_func=fmt,
        max_seq_length=512, tokenizer=tok,
    )
    trainer.train()
    final = os.path.join(out_dir, 'final')
    trainer.model.save_pretrained(final)
    tok.save_pretrained(final)
    print(f'  Adapter saved: {final}')
    return final

def compute_ppl(model, tok, val_file='/content/data/val.jsonl', limit=200):
    model.eval()
    total_loss, total_n = 0.0, 0
    with open(val_file) as f:
        rows = [json.loads(l) for l in f][:limit]
    for ex in rows:
        text = ('### Instruction:\n' + ex['instruction']
                + '\n\n### Response:\n' + ex['output'])
        inp = tok(text, return_tensors='pt', max_length=512, truncation=True).to(model.device)
        with torch.no_grad():
            loss = model(**inp, labels=inp['input_ids']).loss.item()
        total_loss += loss * inp['input_ids'].shape[-1]
        total_n += inp['input_ids'].shape[-1]
    return math.exp(total_loss / total_n)

def gen(model, tok, question, max_new=256):
    prompt = (
        '<s>[INST] <<SYS>>\n' + SYSTEM + '\n<</SYS>>\n\n'
        + '### Instruction:\n' + question + '\n\n### Response:\n'
    )
    inp = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inp, max_new_tokens=max_new, temperature=0.7, top_p=0.9,
            do_sample=True, pad_token_id=tok.eos_token_id
        )
    return tok.decode(out[0][inp['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

print('Helpers ready.')

## Baseline Training (r=16, full dataset)

~40 min on A100. This is the primary adapter used in the qualitative comparison.
Skip and load a pre-trained adapter if you already have one.

In [ ]:
model, tok = load_model(rank=16)
baseline_path = run_train(model, tok, CKPT_BASE + '/r16_full')
baseline_ppl = compute_ppl(model, tok)
print(f'\nFine-tuned perplexity (r=16, full data): {baseline_ppl:.2f}')
del model
torch.cuda.empty_cache()

## Experiment 1: Rank Sweep (r=8 / r=16 / r=64)

Each run: 1000 examples, 3 epochs. Measures the quality-vs-parameter tradeoff.

Expected: r=16 gives ~95% of r=64 quality at 25% the adapter parameters.
The perplexity gap between r=8 and r=64 should be small on this task.

In [ ]:
rank_results = {}

for rank in [8, 16, 64]:
    print('\n' + '='*55)
    print(f'Rank r={rank}')
    m, t = load_model(rank=rank)
    path = run_train(m, t, CKPT_BASE + f'/r{rank}', n=1000)
    ppl = compute_ppl(m, t)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    rank_results[rank] = {'path': path, 'ppl': round(ppl, 2), 'params': n_params}
    print(f'r={rank}: ppl={ppl:.2f}, trainable={n_params:,}')
    del m
    torch.cuda.empty_cache()

print('\n' + '='*55)
print('RANK SWEEP RESULTS')
print(f'{"Rank":<8} {"Perplexity":<14} Trainable Params')
print('-' * 45)
for r in sorted(rank_results):
    v = rank_results[r]
    print(f'r={r:<6} {v["ppl"]:<14} {v["params"]:,}')

## Experiment 2: Data Scaling (n=100 / n=500 / n=1000)

All runs: r=16, 3 epochs. Tests the data requirement threshold.

Expected: meaningful gains 100→500, diminishing returns beyond 500.
Overfitting (train/val loss divergence) appears at epoch 3 for n=100.

In [ ]:
scale_results = {}

for n in [100, 500, 1000]:
    print('\n' + '='*55)
    print(f'Data n={n}')
    m, t = load_model(rank=16)
    path = run_train(m, t, CKPT_BASE + f'/n{n}', n=n)
    ppl = compute_ppl(m, t)
    scale_results[n] = {'path': path, 'ppl': round(ppl, 2)}
    print(f'n={n}: ppl={ppl:.2f}')
    del m
    torch.cuda.empty_cache()

print('\n' + '='*55)
print('DATA SCALING RESULTS (r=16, 3 epochs)')
print(f'{"Examples":<12} Perplexity')
print('-' * 26)
for n in sorted(scale_results):
    print(f'{n:<12} {scale_results[n]["ppl"]}')

## Qualitative Comparison

Side-by-side base model vs fine-tuned. Fine-tuned answers should be more
concise, technically precise, and follow the interview answer style.

In [ ]:
EVAL_PROMPTS = [
    'What is LoRA and why is it more efficient than full fine-tuning?',
    'What is the difference between RAG and fine-tuning for LLMs?',
    'What is the vanishing gradient problem and how is it solved?',
]

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'
)
base_m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto')
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base_tok.pad_token = base_tok.eos_token

ft_m = PeftModel.from_pretrained(base_m, baseline_path)

for q in EVAL_PROMPTS:
    print('\n' + '='*70)
    print('Q:', q)
    print('\n[BASE MODEL]')
    print(gen(base_m, base_tok, q))
    print('\n[FINE-TUNED]')
    print(gen(ft_m, base_tok, q))

del base_m, ft_m
torch.cuda.empty_cache()

## Results Summary

In [ ]:
print('=== RANK SWEEP (1000 examples, 3 epochs) ===')
print(f'{"Rank":<8} {"Perplexity":<14} Trainable Params')
print('-' * 45)
for r in sorted(rank_results):
    v = rank_results[r]
    note = ' <- default' if r == 16 else ''
    print(f'r={r:<6} {v["ppl"]:<14} {v["params"]:,}{note}')

print()
print('=== DATA SCALING (r=16, 3 epochs) ===')
print(f'{"Examples":<12} Perplexity')
print('-' * 26)
for n in sorted(scale_results):
    print(f'{n:<12} {scale_results[n]["ppl"]}')

print()
print(f'Baseline (r=16, full data): ppl={baseline_ppl:.2f}')
print()
print('Fill these numbers into lora-finetune/README.md under "LoRA Rank Experiments".')

## Save / Download Checkpoints

- `USE_DRIVE = True`: already saved to Drive, list paths below.
- `USE_DRIVE = False`: download the baseline adapter as a zip file.

In [ ]:
if USE_DRIVE:
    print('Checkpoints in Drive:', CKPT_BASE)
    for root, dirs, fnames in os.walk(CKPT_BASE):
        for f in fnames:
            if 'adapter' in f:
                print(' ', os.path.join(root, f))
else:
    import zipfile
    from google.colab import files

    zip_path = '/content/adapter_r16_full.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, fnames in os.walk(baseline_path):
            for fname in fnames:
                fp = os.path.join(root, fname)
                zf.write(fp, os.path.relpath(fp, baseline_path))

    print('Downloading:', zip_path)
    files.download(zip_path)
    print()
    print('To run locally:')
    print('  Unzip into lora-finetune/checkpoints/final/')
    print('  python inference.py --adapter ./checkpoints/final --interactive')